# Finetuning LLMs with Ray Train and Deepspeed

This notebook will walk you through finetuning an LLM model for a causal language modeling task using Ray Train and Deepspeed.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook:</b>

<ul>
    <li><b>Part 1: </b>Creating a dataset for LLM training</li>
    <li><b>Part 2: </b>Preprocessing the dataset</li>
    <li><b>Part 3: </b>Defining the training function</li>
    <li><b>Part 4: </b>Launching a distributed training job</li>
</ul>
</div>


## Imports

In [ ]:
!pip install torch numpy tqdm accelerate datasets peft transformers

In [ ]:
import functools
import json
import math
import tempfile
import time

import numpy as np
import ray
import torch
import tqdm
from accelerate import Accelerator, DeepSpeedPlugin
from accelerate.utils import set_seed
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from ray import train
from ray.train import Checkpoint
from ray.train.torch import TorchTrainer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

## Constants

In [ ]:
OPTIM_BETAS = (0.9, 0.999)
OPTIM_EPS = 1e-8
NUM_WARMUP_STEPS = 10
OPTIM_WEIGHT_DECAY = 0.0

## 1. Creating a dataset

We will create a finetuning dataset around the publically available [gsm8k dataset from openai](https://huggingface.co/datasets/openai/gsm8k). This is a dataset of 8.5K high quality linguistically diverse grade school math word problems. The dataset was created to support the task of question answering on basic mathematical problems that require multi-step reasoning.

In [ ]:
dataset = load_dataset("gsm8k", "main")
dataset_splits = {"train": dataset["train"], "test": dataset["test"]}
dataset_splits["train"]

### Aligning with a prompt format
For context, each model has a certain prompt format that it was trained on. Usually, it is best to keep the prompt format the same as the one the model was trained on. 

In this case, we chose to define a custom prompt format with special tokens to demarcate the problem statement and the answer. 

In [ ]:
!mkdir -p /mnt/cluster_storage/finetune_llm/data/

In [ ]:
special_tokens = ["<START_Q>", "<END_Q>", "<START_A>", "<END_A>"]

for key, ds in dataset_splits.items():
    with open(f"/mnt/cluster_storage/finetune_llm/data/{key}.jsonl", "w") as f:
        for item in ds:
            newitem = {}
            newitem["input"] = (
                f"<START_Q>{item['question']}<END_Q>"
                f"<START_A>{item['answer']}<END_A>"
            )
            f.write(json.dumps(newitem) + "\n")

## Preprocessing the dataset

Let's go over the basic preprocessing steps we need to perform on the dataset before we can use it for finetuning.

### Tokenizing the data

The next step in preparing the dataset is to tokenize the data. 

We will use the tokenizer provided by the model to tokenize the data but extend it to include the special tokens we defined in the previous step.

In [ ]:
def get_tokenizer(
    model_name_or_pretrained_path: str, special_tokens: list[str]
) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_pretrained_path, legacy=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.add_tokens(special_tokens, special_tokens=True)
    return tokenizer

In [ ]:
model_id = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = get_tokenizer(model_id, special_tokens)
tokenizer

In [ ]:
token_ids = tokenizer.encode("hello world")
tokens = tokenizer.convert_ids_to_tokens(token_ids)
token_ids, tokens

In [ ]:
labels = tokens[1:]
labels

The labels will simply be the same as the input (shifted by one position to the right by our model). This is because the model will be trained to predict the next token in the sequence, given the previous tokens.

<div class="alert alert-block alert-warning">

<b>Note:</b> For instruction finetuning, it is common to use -100 tokens for the labels matching the non-assistant tokens. This is because we will want the model will ignore the loss for these tokens.

</div>


## Defining the training function

Here is the training function that we will use to train the model.

We will make use of the [accelerate library](https://huggingface.co/docs/accelerate/en/index) as an adapter to interface with both [DeepSpeed](https://www.deepspeed.ai/) and [FSDP](https://pytorch.org/blog/introducing-pytorch-fully-sharded-data-parallel-api/).

In [ ]:
def training_function(config: dict) -> None:
    # [0] Perform some initialization based on the hyperparameters
    # Sample hyper-parameters for learning rate, batch size, seed and a few other HPs
    # ====================================================================================
    lr = float(config["lr"])
    num_epochs = int(config["num_epochs"])
    seed = int(config["seed"])
    batch_size_per_device = int(config["batch_size_per_device"])
    num_devices = int(config["num_devices"])
    gradient_accumulation_steps = int(config["gradient_accumulation_steps"])
    # Get deepspeed config to setup the batch size per device
    deepspeed_plugin = config["deepspeed_plugin"]
    deepspeed_plugin.hf_ds_config.config["train_micro_batch_size_per_gpu"] = (
        batch_size_per_device
    )
    # Tokenizer's special tokens based on data generation
    special_tokens = list(config["special_tokens"])

    # Initialize accelerator
    accelerator = Accelerator(
        deepspeed_plugin=deepspeed_plugin,
        gradient_accumulation_steps=gradient_accumulation_steps,
        mixed_precision=config["mixed_precision"],
    )
    model_id = config["model_id"]

    # Initialize the seed
    set_seed(seed)


    # [1] Prepare the DataLoader for distributed training using Ray Data and Ray Train
    # Shard the datasets among training workerers and move batches to the correct device
    # ====================================================================================
    train_ds = train.get_dataset_shard("train")

    # Get the tokenizer
    tokenizer = get_tokenizer(model_id, special_tokens)
    # Apply the tokenizer via the collate function
    collate_partial = functools.partial(
        tokenize_and_move_to_device,
        tokenizer=tokenizer,
        context_length=config["context_length"],
        device=accelerator.device,
    )
    train_data_loader = train_ds.iter_torch_batches(
        batch_size=batch_size_per_device,
        collate_fn=collate_partial,
        prefetch_batches=config["prefetch_batches"],
    )

    # [2] Initialize the model, optimizer, and learning rate scheduler
    # ====================================================================================
    # Create model object
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        use_cache=False,
        attn_implementation="flash_attention_2",
    )
    # Resize the token embeddings to include the special tokens
    model.resize_token_embeddings(len(tokenizer))

    # Apply LoRA
    lora_config = LoraConfig(**config["lora_config"])
    model.enable_input_require_grads()
    model = get_peft_model(model, lora_config)

    # Enable gradient checkpointing to save on memory - trade-off with speed
    model.gradient_checkpointing_enable()

    # Set model to training mode
    model.train()

    # Instantiate Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=OPTIM_BETAS,
        weight_decay=OPTIM_WEIGHT_DECAY,
        eps=OPTIM_EPS,
    )

    # Instantiate Learning rate scheduler
    num_steps_per_epoch = math.ceil(config["num_samples"] / batch_size_per_device)
    total_training_steps = (
        num_steps_per_epoch * num_epochs // gradient_accumulation_steps
    )
    lr_scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=NUM_WARMUP_STEPS * num_devices,
        num_training_steps=total_training_steps * num_devices,
    )

    # [3] Prepare everything
    # We use accelerate directly to prepare the model, optimizer, and lr_scheduler
    # There is no specific order to remember, we just need to unpack the objects in the
    # same order we gave them to the prepare method.
    # ====================================================================================
    model, optimizer, lr_scheduler = accelerator.prepare(model, optimizer, lr_scheduler)

    # [4] Start training
    # This is a standard PyTorch training loop, but we use the accelerator to handle
    # running it for us and perform proper gradient accumulation
    # ====================================================================================
    if accelerator.is_main_process:
        print("Starting training ...")
        print("Number of batches on main process", num_steps_per_epoch)

    for epoch in range(num_epochs):
        loss_sum = torch.tensor(0.0).to(accelerator.device)
        for step, batch in tqdm.tqdm(
            enumerate(train_data_loader, start=1),
            total=num_steps_per_epoch,
        ):
            # gradient accumulation
            with accelerator.accumulate(model):
                # forward pass
                s_fwd = time.perf_counter()
                outputs = model(**batch)
                loss = outputs.loss
                loss_sum += loss.item()
                e_fwd = time.perf_counter()
                fwd_time = e_fwd - s_fwd

                # backward pass
                s_bwd = time.perf_counter()
                accelerator.backward(loss)
                e_bwd = time.perf_counter()
                bwd_time = e_bwd - s_bwd

                # update weights
                s_opt_step = time.perf_counter()
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                e_opt_step = time.perf_counter()
                opt_step_time = e_opt_step - s_opt_step

            if accelerator.is_main_process:
                print(f"Epoch {epoch} batch loss on main process: {loss.item() / step}")

            train.report(
                metrics={
                    "epoch": epoch,
                    "iteration": step,
                    "avg_train_loss_epoch": None,
                    "num_iterations": step + 1,
                    "fwd_time": fwd_time,
                    "bwd_time": bwd_time,
                    "opt_step_time": opt_step_time,
                    "learning_rate": lr_scheduler.get_lr()[0],
                }
            )

        # [5] Save the model checkpoint to the configured persistent storage location
        # ================================================================================
        metrics = {
            "epoch": epoch,
            "iteration": step,
            "avg_train_loss_epoch": loss_sum.item() / (step + 1),
            "num_iterations": step + 1,
            "fwd_time": fwd_time,
            "bwd_time": bwd_time,
            "opt_step_time": opt_step_time,
            "learning_rate": lr_scheduler.get_lr()[0],
        }
        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            print(f"Saving the model locally at {temp_checkpoint_dir}")
            accelerator.wait_for_everyone()

            if accelerator.is_main_process:
                print("Saving tokenizer and config.")
                tokenizer.save_pretrained(temp_checkpoint_dir)

            accelerator.wait_for_everyone()

            # Aggregating the model state_dict to the main process
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(
                temp_checkpoint_dir,
                is_main_process=accelerator.is_main_process,
                save_function=accelerator.save,
                safe_serialization=True,
                state_dict=accelerator.get_state_dict(model),
            )
            accelerator.wait_for_everyone()
            checkpoint = (
                Checkpoint.from_directory(temp_checkpoint_dir)
                if accelerator.is_main_process
                else None
            )
            train.report(metrics=metrics, checkpoint=checkpoint)
            accelerator.wait_for_everyone()

Here is a preprocessing that we choose to apply on the training worker process

In [ ]:
def tokenize_and_move_to_device(
    batch: dict[str, np.ndarray],
    tokenizer: AutoTokenizer,
    context_length: int,
    device: str,
) -> dict[str, torch.Tensor]:
    out_batch = tokenizer(
        list(batch["input"]),
        padding="max_length", # Note: you can use 'longest' to save on memory if needed
        max_length=context_length,
        truncation=True,
        return_tensors="pt",
    )
    out_batch["labels"] = out_batch["input_ids"].clone()
    for key, values in out_batch.items():
        out_batch[key] = values.to(device)
    return out_batch

<div class="alert alert-block alert-warning">

<b>Note:</b> The `collate_fn` parameter in `iter_torch_batches` allows you to transform data before feeding it to the model. This operation happens locally in the training workers. Avoid adding a heavy transformation in this function as it may become the bottleneck. Instead, apply the transformation with `map_batches` before passing the dataset to the `Trainer`.

In the context of LLM Finetuning, if you are performing *direct preference optimization (DPO)* or *knowledge distillation*, you will need to generate inference from a reference/teacher model as part of the training loop. You should not use the `collate_fn` to generate the inference as it will require placing the reference model on the same GPU node as the training worker drastically affecting the available GPU memory and throughput. Instead, you should generate the inference asynchronously by passing in a ray dataset that maps the input to the reference model's inference.

</div>


### Construct our input dataset pipeline

In [ ]:
train_ds = ray.data.read_json("/mnt/cluster_storage/finetune_llm/data/train.jsonl")

## Launching a distributed training job

In [ ]:
config = {
    "lr": 5e-6,
    "num_epochs": 1,
    "seed": 42,
    "batch_size_per_device": 16,
    "num_devices": 2,
    "gradient_accumulation_steps": 1,
    "mixed_precision": "bf16",
    "model_id": "meta-llama/Llama-2-7b-chat-hf",
    "context_length": 512,
    "prefetch_batches": 2,
}

with open("./lora_config.json", "r") as json_file:
    lora_config = json.load(json_file)

config["lora_config"] = lora_config
config["deepspeed_plugin"] = DeepSpeedPlugin(hf_ds_config="./deepspeed_config.json")
config["special_tokens"] = special_tokens
config["num_samples"] = train_ds.count()

## Create the trainer

Next, we will create the trainer object that will be used to train the model.

In [ ]:
storage_path = f"/mnt/cluster_storage/ft_llms_with_deepspeed/{config['model_id']}"

trainer = TorchTrainer(
    training_function,
    train_loop_config=config,
    run_config=train.RunConfig(storage_path=storage_path),
    scaling_config=train.ScalingConfig(
        num_workers=config["num_devices"],
        use_gpu=True,
        resources_per_worker={"GPU": 1},
    ),
    datasets={"train": train_ds},
    dataset_config=ray.train.DataConfig(datasets_to_split=["train", "valid"]),
)

## Run the training job

We proceed to call `trainer.fit` to start the training job.

In [ ]:
result = trainer.fit()

print(result)